# Experiment: TCP Content-Aware Adaptive Batching

Objective:
- Treat TCP/HTTP2 flush-batching as a `Policy Learning / Adaptive Control` problem.
- Follow the current direction in `progress_notes.md`: `Heuristic baseline -> ML adaptive policy -> RL adaptive transmission policy`.
- Reproduce heuristic comparison, fixed sweep, oracle selection, and `RandomForestRegressor`-based `ml_regression_adaptive` evaluation in one notebook.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import math
import subprocess
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
}

missing = [pkg for module, pkg in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

SEED = 20260309
MSS_BYTES = 1460
FIXED_BATCH_GRID = [512, 2048, 8192, 32768, 65536]
FIXED_FLUSH_GRID = [0.0, 2.0, 5.0, 10.0, 20.0, 50.0]

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 120


def resolve_repo_root() -> Path:
    markers = ("progress_notes.md", "init_plan.md")
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if all((candidate / marker).exists() for marker in markers):
            return candidate
        if (candidate / ".git").exists():
            return candidate
    return Path.cwd().resolve()


def finalize_figure(fig) -> None:
    try:
        from IPython import get_ipython
        shell = get_ipython()
    except Exception:
        shell = None
    if shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell":
        plt.show()
    else:
        plt.close(fig)


REPO_ROOT = resolve_repo_root()
OUTPUT_DIR = REPO_ROOT / "output" / "jupyter-notebook"
ASSET_DIR = OUTPUT_DIR / "assets"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ASSET_DIR.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "python": sys.version.split()[0],
    "seed": SEED,
    "repo_root": str(REPO_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "assets_dir": str(ASSET_DIR),
}, indent=2))


## Baseline Scope

- 워크로드: `static_file`, `dynamic_stream`
- baseline 정책: `immediate`, `fixed_batch(8192B/10ms)`, `heuristic_adaptive`
- 대표 지표: `latency_mean_ms`, `latency_p95_ms`, `throughput_mbps`, `goodput_bytes`, `flush_count`, `mean_batch_size_bytes`, `staleness_penalty`
- 본 노트북은 대표 시나리오 비교까지 먼저 고정하고, full sweep과 ML 학습은 다음 셀 추가로 확장한다.


In [2]:
@dataclass(frozen=True)
class WorkloadConfig:
    content_type: str
    file_size_bytes: Optional[int] = None
    chunk_size_bytes: Optional[int] = None
    message_size_bytes: Optional[int] = None
    interarrival_ms: Optional[float] = None
    change_rate: str = "low"
    freshness_budget_ms: Optional[float] = None
    rtt_ms: float = 10.0
    bandwidth_mbps: float = 20.0
    delayed_ack_ms: float = 10.0
    generation_gap_ms: float = 0.05
    sample_messages: int = 240

@dataclass(frozen=True)
class PolicyConfig:
    name: str
    batch_bytes: Optional[int] = None
    flush_interval_ms: Optional[float] = None

REFERENCE_FIXED = PolicyConfig("fixed_batch", 8192, 10.0)

REFERENCE_CASES = [
    WorkloadConfig(
        content_type="static_file",
        file_size_bytes=16 * 1024 * 1024,
        chunk_size_bytes=4096,
        rtt_ms=30,
        bandwidth_mbps=20,
        delayed_ack_ms=40,
    ),
    WorkloadConfig(
        content_type="dynamic_stream",
        message_size_bytes=1200,
        interarrival_ms=10,
        change_rate="high",
        freshness_budget_ms=25,
        rtt_ms=50,
        bandwidth_mbps=5,
        delayed_ack_ms=40,
        sample_messages=300,
    ),
]

pd.DataFrame([asdict(cfg) for cfg in REFERENCE_CASES]).fillna("-")


,content_type,file_size_bytes,chunk_size_bytes,message_size_bytes,interarrival_ms,change_rate,freshness_budget_ms,rtt_ms,bandwidth_mbps,delayed_ack_ms,generation_gap_ms,sample_messages
0,static_file,16777216.0,4096.0,-,-,low,-,30,20,40,0.05,240
1,dynamic_stream,-,-,1200.0,10.0,high,25.0,50,5,40,0.05,300


In [ ]:
def scenario_id(cfg: WorkloadConfig) -> str:
    if cfg.content_type == "static_file":
        return f"static_{cfg.chunk_size_bytes}_{cfg.rtt_ms}_{cfg.bandwidth_mbps}_{cfg.delayed_ack_ms}"
    return f"stream_{cfg.message_size_bytes}_{cfg.interarrival_ms}_{cfg.change_rate}_{cfg.rtt_ms}_{cfg.bandwidth_mbps}_{cfg.freshness_budget_ms}"


def generate_workload(cfg: WorkloadConfig) -> pd.DataFrame:
    local_rng = np.random.default_rng(SEED + sum(ord(ch) for ch in scenario_id(cfg)) % 10000)
    if cfg.content_type == "static_file":
        count = int(math.ceil(cfg.file_size_bytes / cfg.chunk_size_bytes))
        payloads = np.full(count, cfg.chunk_size_bytes, dtype=int)
        payloads[-1] -= max(0, count * cfg.chunk_size_bytes - cfg.file_size_bytes)
        event_time_ms = np.arange(count, dtype=float) * cfg.generation_gap_ms
        freshness = np.full(count, np.inf, dtype=float)
    else:
        jitter = 0.10 if cfg.change_rate == "low" else 0.25
        gaps = local_rng.normal(cfg.interarrival_ms, max(cfg.interarrival_ms * jitter, 0.5), size=cfg.sample_messages)
        gaps = np.clip(gaps, 1.0, None)
        event_time_ms = np.cumsum(gaps) - gaps[0]
        payload_noise = local_rng.uniform(0.8, 1.2, size=cfg.sample_messages)
        payloads = np.maximum(64, np.rint(cfg.message_size_bytes * payload_noise).astype(int))
        freshness = np.full(cfg.sample_messages, cfg.freshness_budget_ms, dtype=float)
    return pd.DataFrame({
        "event_time_ms": event_time_ms,
        "payload_bytes": payloads,
        "freshness_budget_ms": freshness,
    })


def build_policy_features(cfg: WorkloadConfig, events: Optional[pd.DataFrame] = None) -> dict:
    if events is None:
        events = generate_workload(cfg)

    payload_series = events["payload_bytes"].astype(float)
    if len(events) > 1:
        inter_arrival_series = events["event_time_ms"].diff().iloc[1:]
        inter_arrival_time_ms = float(inter_arrival_series.mean())
    else:
        inter_arrival_time_ms = float(cfg.generation_gap_ms if cfg.content_type == "static_file" else cfg.interarrival_ms)

    message_size = float(payload_series.mean())
    if cfg.content_type == "static_file":
        decision_window_ms = float(cfg.rtt_ms)
    else:
        decision_window_ms = float(min(cfg.freshness_budget_ms, cfg.rtt_ms))

    queue_messages = max(1.0, decision_window_ms / max(inter_arrival_time_ms, 1e-6))
    queue_size = float(message_size * queue_messages)
    elapsed_time_since_last_flush = float(min(decision_window_ms, max(inter_arrival_time_ms, 0.0)))

    return {
        "content_type": cfg.content_type,
        "change_rate": cfg.change_rate,
        "message_size": message_size,
        "inter_arrival_time": inter_arrival_time_ms,
        "queue_size": queue_size,
        "elapsed_time_since_last_flush": elapsed_time_since_last_flush,
        "estimated_rtt": float(cfg.rtt_ms),
        "bandwidth_mbps": float(cfg.bandwidth_mbps),
        "delayed_ack_ms": float(cfg.delayed_ack_ms),
        "freshness_budget_ms": float(cfg.freshness_budget_ms or 0.0),
    }


def snap(value: float, grid: List[float]) -> float:
    return min(grid, key=lambda candidate: abs(candidate - value))


def resolve_policy(cfg: WorkloadConfig, policy: PolicyConfig) -> Tuple[int, float]:
    if policy.name == "immediate":
        return 1, 0.0
    if policy.name == "fixed_batch":
        return int(policy.batch_bytes), float(policy.flush_interval_ms)
    if cfg.content_type == "static_file":
        bdp = (cfg.bandwidth_mbps * 1_000_000 / 8.0) * (cfg.rtt_ms / 1000.0)
        batch = max(4 * MSS_BYTES, 0.5 * bdp)
        flush_ms = min(0.5 * cfg.rtt_ms, cfg.delayed_ack_ms)
    else:
        batch = min(2 * MSS_BYTES, cfg.message_size_bytes * 4.0)
        flush_ms = min(0.5 * cfg.freshness_budget_ms, cfg.interarrival_ms * 1.5)
        if cfg.change_rate == "high":
            batch *= 0.5
            flush_ms *= 0.5
    return int(max(1, snap(batch, FIXED_BATCH_GRID))), float(max(0.0, snap(flush_ms, FIXED_FLUSH_GRID)))


def run_simulation(cfg: WorkloadConfig, policy: PolicyConfig) -> dict:
    events = generate_workload(cfg)
    policy_features = build_policy_features(cfg, events)
    batch_target, flush_limit = resolve_policy(cfg, policy)
    arrivals, freshness, payloads = [], [], []
    queue_bytes = 0
    batch_start = None
    latencies, stale, batch_sizes = [], [], []
    queue_samples, flush_age_samples = [], []
    total_payload, flush_count = 0, 0
    first_arrival = float(events["event_time_ms"].iloc[0]) if not events.empty else 0.0
    last_completion = first_arrival

    def flush(now: float) -> None:
        nonlocal arrivals, freshness, payloads, queue_bytes, batch_start, total_payload, flush_count, last_completion
        if not payloads:
            return
        payload = int(sum(payloads))
        tx_time = (payload * 8.0) / (cfg.bandwidth_mbps * 1_000_000.0) * 1000.0
        ack_penalty = min(cfg.delayed_ack_ms, 0.25 * cfg.rtt_ms) if payload < MSS_BYTES else 0.0
        completion = now + tx_time + (cfg.rtt_ms / 2.0) + ack_penalty
        for arrival, budget in zip(arrivals, freshness):
            latency = completion - arrival
            latencies.append(latency)
            stale.append(max(0.0, latency - budget) if math.isfinite(budget) else 0.0)
        batch_sizes.append(payload)
        total_payload += payload
        flush_count += 1
        last_completion = completion
        arrivals, freshness, payloads = [], [], []
        queue_bytes = 0
        batch_start = None

    for row in events.itertuples(index=False):
        t = float(row.event_time_ms)
        if payloads and flush_limit > 0 and batch_start is not None and t - batch_start >= flush_limit:
            flush(batch_start + flush_limit)
        if not payloads:
            batch_start = t
        arrivals.append(t)
        freshness.append(float(row.freshness_budget_ms))
        payloads.append(int(row.payload_bytes))
        queue_bytes += int(row.payload_bytes)
        queue_samples.append(queue_bytes)
        flush_age_samples.append(0.0 if batch_start is None else t - batch_start)
        if flush_limit == 0.0 or queue_bytes >= batch_target:
            flush(t)
    if payloads:
        final_time = max(float(events["event_time_ms"].iloc[-1]), batch_start + flush_limit if flush_limit > 0 else float(events["event_time_ms"].iloc[-1]))
        flush(final_time)

    duration = max(last_completion - first_arrival, 1e-6)
    latency_array = np.asarray(latencies, dtype=float)
    stale_array = np.asarray(stale, dtype=float)
    batch_array = np.asarray(batch_sizes, dtype=float)
    queue_array = np.asarray(queue_samples, dtype=float)
    flush_age_array = np.asarray(flush_age_samples, dtype=float)
    return {
        **asdict(cfg),
        **policy_features,
        "scenario_id": scenario_id(cfg),
        "policy": policy.name,
        "resolved_batch_bytes": batch_target,
        "resolved_flush_interval_ms": flush_limit,
        "latency_mean_ms": float(latency_array.mean()),
        "latency_p95_ms": float(np.quantile(latency_array, 0.95)),
        "throughput_mbps": float((total_payload * 8.0) / duration / 1000.0),
        "goodput_bytes": float(total_payload),
        "flush_count": int(flush_count),
        "syscall_count": int(flush_count),
        "mean_batch_size_bytes": float(batch_array.mean()),
        "mean_queue_size_bytes": float(queue_array.mean()) if len(queue_array) else 0.0,
        "max_queue_size_bytes": float(queue_array.max()) if len(queue_array) else 0.0,
        "mean_elapsed_time_since_last_flush_ms": float(flush_age_array.mean()) if len(flush_age_array) else 0.0,
        "staleness_penalty": float(stale_array.sum()),
    }


POLICIES = [
    PolicyConfig("immediate"),
    REFERENCE_FIXED,
    PolicyConfig("heuristic_adaptive"),
]

reference_results = pd.DataFrame([
    run_simulation(cfg, policy)
    for cfg in REFERENCE_CASES
    for policy in POLICIES
])

reference_summary = reference_results[[
    "content_type",
    "policy",
    "latency_mean_ms",
    "latency_p95_ms",
    "throughput_mbps",
    "goodput_bytes",
    "flush_count",
    "syscall_count",
    "mean_batch_size_bytes",
    "staleness_penalty",
]].copy()

reference_summary


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)
sns.barplot(data=reference_summary, x="policy", y="throughput_mbps", hue="content_type", ax=axes[0])
axes[0].set_title("Reference Throughput Comparison")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(data=reference_summary, x="policy", y="latency_p95_ms", hue="content_type", ax=axes[1])
axes[1].set_title("Reference p95 Latency Comparison")
axes[1].tick_params(axis="x", rotation=15)
if axes[1].legend_ is not None:
    axes[1].legend_.remove()

figure_path = ASSET_DIR / "reference_policy_overview.png"
fig.savefig(figure_path, bbox_inches="tight")
finalize_figure(fig)

print(f"Saved {figure_path}")
reference_summary


Saved output\jupyter-notebook\assets\reference_policy_overview.png


C:\Users\Administrator\AppData\Local\Temp\ipykernel_31424\685377020.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,content_type,policy,latency_mean_ms,latency_p95_ms,throughput_mbps,goodput_bytes,flush_count,syscall_count,mean_batch_size_bytes,staleness_penalty
0,static_file,immediate,16.638400,16.63840,606.254564,16777216.0,4096,4096,4096.000000,0.000000
1,static_file,fixed_batch,18.301800,18.32680,601.800896,16777216.0,2048,2048,8192.000000,0.000000
2,static_file,heuristic_adaptive,28.282200,28.45720,576.395009,16777216.0,512,512,32768.000000,0.000000
3,dynamic_stream,immediate,39.445147,39.78328,0.981242,364715.0,300,300,1215.716667,4333.544000
4,dynamic_stream,fixed_batch,40.303143,49.73208,0.977953,364715.0,204,204,1787.818627,4590.942844
5,dynamic_stream,heuristic_adaptive,43.996838,44.78328,0.979595,364715.0,295,295,1236.322034,5699.051297


## Extended Steps (진행 반영)

- [x] `fixed_sweep(scenarios)`와 `pick_oracle(sweep_df)` 구현
- [x] `feature_frame`, `RandomForestRegressor` 기반 `ml_regression_adaptive` 평가 셀 추가
- [x] heatmap, Pareto scatter, CSV export를 `output/jupyter-notebook/assets/` 아래에 정리
- [ ] RL 실험용 `rl/env.py` 초안 추가

참고: 실험 축이 바뀌면 먼저 [`init_plan.md`](../../init_plan.md)를 수정하고, 그 다음 노트북 상단 설정을 맞춘다.


## Full Sweep, Oracle, ML Adaptive, Visualization

아래 셀은 고정 배치 sweep, oracle 추출, ML 기반 adaptive policy 평가, 시각화/CSV export를 한 번에 수행한다.



In [ ]:
SCENARIOS = [
    WorkloadConfig(content_type="static_file", file_size_bytes=16 * 1024 * 1024, chunk_size_bytes=chunk, rtt_ms=rtt, bandwidth_mbps=bw, delayed_ack_ms=ack)
    for chunk in [1024, 4096, 16384]
    for rtt in [10, 50]
    for bw in [5, 20]
    for ack in [10, 40]
] + [
    WorkloadConfig(content_type="dynamic_stream", message_size_bytes=msg, interarrival_ms=ia, change_rate=rate, freshness_budget_ms=fb, rtt_ms=rtt, bandwidth_mbps=bw, delayed_ack_ms=40, sample_messages=280)
    for msg in [256, 1200]
    for ia in [5, 20]
    for rate in ["low", "high"]
    for fb in [25, 75]
    for rtt in [10, 50]
    for bw in [5, 20]
]

OBJECTIVE_WEIGHTS = {
    "static_file": {
        "throughput_mbps": 0.55,
        "latency_p95_ms": 0.20,
        "syscall_count": 0.15,
        "staleness_penalty": 0.10,
    },
    "dynamic_stream": {
        "throughput_mbps": 0.30,
        "latency_p95_ms": 0.40,
        "staleness_penalty": 0.20,
        "syscall_count": 0.10,
    },
}
REVERSE_SCORE_METRICS = {"latency_p95_ms", "staleness_penalty", "syscall_count"}


def fixed_sweep(scenarios: List[WorkloadConfig]) -> pd.DataFrame:
    rows = []
    for cfg in scenarios:
        for batch in FIXED_BATCH_GRID:
            for flush_ms in FIXED_FLUSH_GRID:
                result = run_simulation(cfg, PolicyConfig("fixed_batch", batch, flush_ms))
                result["batch_bytes"] = int(batch)
                result["flush_interval_ms"] = float(flush_ms)
                rows.append(result)
    return pd.DataFrame(rows)


def build_score_bounds(df: pd.DataFrame) -> Dict[str, Tuple[float, float]]:
    return {
        metric: (float(df[metric].min()), float(df[metric].max()))
        for metric in OBJECTIVE_WEIGHTS[df["content_type"].iloc[0]]
    }


def normalize_metric(value: float, lower: float, upper: float, reverse: bool = False) -> float:
    if upper > lower:
        normalized = (value - lower) / (upper - lower)
    else:
        normalized = 0.5
    normalized = float(np.clip(normalized, 0.0, 1.0))
    return 1.0 - normalized if reverse else normalized


def score_rows(df: pd.DataFrame, bounds: Dict[str, Tuple[float, float]]) -> pd.DataFrame:
    content_type = df["content_type"].iloc[0]
    scored = df.copy()
    objective = np.zeros(len(scored), dtype=float)
    for metric, weight in OBJECTIVE_WEIGHTS[content_type].items():
        reverse = metric in REVERSE_SCORE_METRICS
        lower, upper = bounds[metric]
        normalized = scored[metric].apply(lambda value: normalize_metric(float(value), lower, upper, reverse=reverse))
        scored[f"{metric}_score"] = normalized
        objective += weight * normalized.to_numpy(dtype=float)
    scored["objective_score"] = objective
    return scored


def score_candidate_row(candidate: dict, bounds: Dict[str, Tuple[float, float]]) -> float:
    candidate_df = score_rows(pd.DataFrame([candidate]), bounds)
    return float(candidate_df.iloc[0]["objective_score"])


def pick_oracle(sweep_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Dict[str, Tuple[float, float]]]]:
    scored_groups = []
    bounds_by_scenario = {}
    for scenario_key, group in sweep_df.groupby("scenario_id", sort=False):
        group = group.reset_index(drop=True)
        bounds = build_score_bounds(group)
        scored_group = score_rows(group, bounds)
        scored_groups.append(scored_group)
        bounds_by_scenario[scenario_key] = bounds
    scored = pd.concat(scored_groups, ignore_index=True)
    idx = scored.groupby("scenario_id")["objective_score"].idxmax()
    oracle = scored.loc[idx].copy().reset_index(drop=True)
    return scored, oracle, bounds_by_scenario


sweep_df = fixed_sweep(SCENARIOS)
scored_sweep_df, oracle_df, score_bounds = pick_oracle(sweep_df)

scenario_frame = pd.DataFrame([
    {
        **asdict(cfg),
        **build_policy_features(cfg),
        "scenario_id": scenario_id(cfg),
    }
    for cfg in SCENARIOS
])
feature_cols = [
    "content_type",
    "change_rate",
    "message_size",
    "inter_arrival_time",
    "queue_size",
    "elapsed_time_since_last_flush",
    "estimated_rtt",
    "bandwidth_mbps",
    "delayed_ack_ms",
    "freshness_budget_ms",
]
train_df = scenario_frame.merge(oracle_df[["scenario_id", "batch_bytes", "flush_interval_ms", "objective_score"]], on="scenario_id", how="left")
feature_frame = pd.get_dummies(train_df[feature_cols], columns=["content_type", "change_rate"], dummy_na=True).fillna(0.0)

scenario_ids = train_df["scenario_id"].tolist()
train_ids, holdout_ids = train_test_split(scenario_ids, test_size=0.3, random_state=SEED)
train_mask = train_df["scenario_id"].isin(train_ids)
holdout_mask = train_df["scenario_id"].isin(holdout_ids)

batch_model = RandomForestRegressor(n_estimators=200, random_state=SEED)
flush_model = RandomForestRegressor(n_estimators=200, random_state=SEED + 1)
batch_model.fit(feature_frame.loc[train_mask], train_df.loc[train_mask, "batch_bytes"])
flush_model.fit(feature_frame.loc[train_mask], train_df.loc[train_mask, "flush_interval_ms"])


def to_ml_policy(row_features: pd.DataFrame) -> PolicyConfig:
    pred_batch = float(batch_model.predict(row_features)[0])
    pred_flush = float(flush_model.predict(row_features)[0])
    return PolicyConfig(
        name="ml_regression_adaptive",
        batch_bytes=int(max(1, snap(pred_batch, [float(x) for x in FIXED_BATCH_GRID]))),
        flush_interval_ms=float(max(0.0, snap(pred_flush, FIXED_FLUSH_GRID))),
    )


ml_rows = []
for _, row in train_df.loc[holdout_mask].iterrows():
    scenario_key = row["scenario_id"]
    cfg = next(cfg for cfg in SCENARIOS if scenario_id(cfg) == scenario_key)
    row_features = feature_frame.loc[[row.name]]
    ml_policy = to_ml_policy(row_features)
    ml_result = run_simulation(cfg, ml_policy)
    heur_result = run_simulation(cfg, PolicyConfig("heuristic_adaptive"))

    bounds = score_bounds[scenario_key]
    oracle_row = oracle_df.loc[oracle_df["scenario_id"] == scenario_key].iloc[0]
    reference_scores = scored_sweep_df.loc[scored_sweep_df["scenario_id"] == scenario_key, "objective_score"]
    oracle_score = float(oracle_row["objective_score"])

    for comparison, result in [
        ("ml_regression_adaptive", ml_result),
        ("heuristic_adaptive", heur_result),
    ]:
        candidate_score = score_candidate_row(result, bounds)
        ml_rows.append({
            **result,
            "comparison": comparison,
            "oracle_score": oracle_score,
            "candidate_score": candidate_score,
            "oracle_score_gap": oracle_score - candidate_score,
            "candidate_rank_vs_fixed_sweep": int(1 + (reference_scores > candidate_score).sum()),
        })

ml_eval_df = pd.DataFrame(ml_rows)

heatmap_source = scored_sweep_df.groupby(["content_type", "batch_bytes", "flush_interval_ms"], as_index=False)["objective_score"].mean()
fig, axes = plt.subplots(1, 2, figsize=(18, 6), constrained_layout=True)
for ax, content_type in zip(axes, ["static_file", "dynamic_stream"]):
    pivot = heatmap_source[heatmap_source["content_type"] == content_type].pivot(index="batch_bytes", columns="flush_interval_ms", values="objective_score")
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="viridis", ax=ax)
    ax.set_title(f"fixed_batch heatmap ({content_type})")

heatmap_path = ASSET_DIR / "fixed_batch_heatmap.png"
fig.savefig(heatmap_path, bbox_inches="tight")
finalize_figure(fig)

pareto_df = pd.concat([
    reference_results.assign(comparison=reference_results["policy"]),
    ml_eval_df,
], ignore_index=True, sort=False)

fig, ax = plt.subplots(1, 1, figsize=(10, 7), constrained_layout=True)
sns.scatterplot(
    data=pareto_df,
    x="latency_p95_ms",
    y="throughput_mbps",
    hue="comparison",
    style="content_type",
    ax=ax,
)
ax.set_title("Policy Pareto Scatter (lower latency / higher throughput)")
pareto_path = ASSET_DIR / "policy_pareto_scatter.png"
fig.savefig(pareto_path, bbox_inches="tight")
finalize_figure(fig)

reference_summary.to_csv(ASSET_DIR / "reference_summary.csv", index=False)
scored_sweep_df.to_csv(ASSET_DIR / "fixed_sweep_results.csv", index=False)
oracle_df.to_csv(ASSET_DIR / "oracle_selection.csv", index=False)
ml_eval_df.to_csv(ASSET_DIR / "ml_eval_results.csv", index=False)

print("saved:")
print(f"- {heatmap_path}")
print(f"- {pareto_path}")
print(f"- {ASSET_DIR / 'reference_summary.csv'}")
print(f"- {ASSET_DIR / 'fixed_sweep_results.csv'}")
print(f"- {ASSET_DIR / 'oracle_selection.csv'}")
print(f"- {ASSET_DIR / 'ml_eval_results.csv'}")

ml_eval_df[["scenario_id", "comparison", "candidate_score", "oracle_score_gap", "candidate_rank_vs_fixed_sweep"]].head(12)
